# LLM 제품 사전 적용

규칙 버전 제품 사전(`product_dictionary`)의 언급에 LLM 제품 사전(`product_llm` 23절이 내보낸 표)을 붙여 **그래프에 넣을 제품 표**를 만든다(`graphDB.md` 6.3).

- 언급마다 LLM이 정한 제품으로 잇는다. 섞인 노드는 나뉘고(웨이퍼 → 반도체 웨이퍼 / 웨하스), 여러 품목이 적힌 언급은 제품 여러 개로 간다("레미콘, 이형철근" → 2개).
- **LLM 결과가 없는 언급**(새로 들어온 기사·공시, 관계에 쓰이지 않아 LLM에 보내지 않은 뉴스 언급)은 규칙 노드로 대신 잇는다.
  - 그 규칙 노드의 다른 언급이 모두 같은 LLM 제품으로 갔으면 그 제품으로 잇는다.
  - 아니면(규칙 노드가 새것이거나 LLM이 나눈 노드) 규칙 노드를 **임시 제품**(`PR:`, `pending_llm=True`)으로 둔다. 다음 LLM 실행 때 정리된다.
- 계층: 제품 → 상위 제품 → 제품군(`PF:`). 후보 하위 제품(`PC:`)은 상위 제품 아래에 두고 검색어로만 쓴다.
- **브랜드 → 제품**: LLM 3단계 결과(브랜드 언급마다 이어질 제품)를 브랜드 언급에 붙여, 브랜드 노드마다 이어질 제품(제품을 못 찾았으면 제품군)을 모은다(그래프의 `OF_PRODUCT`). 브랜드 노드 자체는 규칙 사전 그대로다.
- 서버나 LLM에 요청하지 않는다.

**입력**

| 파일 | 쓰는 것 |
|---|---|
| `source/graph/products.csv`, `product_mentions.csv` | 규칙 사전의 제품 노드와 언급(`product_dictionary`) |
| `source/products/llm/llm_mention_map.csv`, `llm_products.csv`, `llm_candidates.csv` | LLM 사전: 언급 → 제품, 제품, 후보 하위 제품(`product_llm` 23절) |
| `source/products/llm/families_final.csv` | 제품군 목록 |
| `source/graph/brands.csv`, `source/products/llm/stage3_brands_product_brand_v3.csv` | 규칙 사전의 브랜드 노드, LLM 3단계 결과(브랜드 언급 → 종류·제품군·제품, `product_llm` 28절) |
| `source/dart/parsed/companies.csv`, `source/news/final/news_final.json` | 언급의 업종(회사의 WI26 업종, 기사의 업종) |

**출력** (`source/graph/`, 내용이 바뀔 때만 쓴다)

| 파일 | 한 줄 |
|---|---|
| `products_final.csv` | 제품 노드 하나: ID, 단계(제품군 / 제품 / 후보 / 임시), 이름, 제품군, 종류, 규격, 검색어, 만드는 국내 상장사 수, 뉴스 기사 수, 관련 업종, 일반적인 제품인지 |
| `product_mentions_final.csv` | 언급 하나 → 제품 하나(여러 품목 언급은 여러 줄). 어떻게 이었는지(`how`) |
| `product_hierarchy.csv` | 하위 → 상위 한 줄(그래프의 `SUBTYPE_OF`) |
| `brands_final.csv` | 브랜드 노드 하나: 규칙 사전의 열 + 종류(`brand_kind`: 제품 브랜드 / 서비스 브랜드 / 회사·그룹 이름 / 브랜드 아님) |
| `brand_products.csv` | 브랜드 → 제품(또는 제품군) 한 줄(그래프의 `OF_PRODUCT`): 이어진 언급 수, 어떻게 찾았는지 |

**한눈에**

| | |
|---|---|
| 처리 순서에서 | `product_dictionary`(규칙 사전) → `product_llm`(LLM 실행·내보내기) → **`product_apply`** → `graph_db` |
| 외부 요청·비용 | 없음 |
| 다시 실행 | 언제든 된다 |
| 처리 순서 | 입력 → 언급 → 제품 → 제품 노드(제품군·제품·후보·임시) → 계층 → 브랜드 → 제품 → 저장 → 확인 |

## 1. 설정

In [1]:
# ── 설정 ──
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'
GRAPH = ROOT / 'source' / 'graph'
LLM = ROOT / 'source' / 'products' / 'llm'
MKEY = ['book', 'source', 'doc', 'field', 'ref', 'name', 'name_en']   # 언급 하나를 가리키는 열
MAX_TERMS = 20                 # 제품 하나의 검색어 최대 수(검색어 노트북이 다시 고른다)
GENERIC_MAKERS = 10            # 국내 상장사 이만큼 이상이 만들면 일반적인 제품(규칙 사전과 같은 기준)
BRAND_FILE = LLM / 'stage3_brands_product_brand_v3.csv'   # LLM 3단계 결과로 쓰기로 한 v3
split = lambda s: [x for x in (s or '').split(' | ') if x]
low = lambda s: re.sub(r'\s+', ' ', (s or '').strip().lower())
HANGUL = re.compile('[가-힣]')


def save_csv(df, path):
    """내용이 바뀔 때만 쓴다(iCloud 사본 방지)."""
    text = df.to_csv(index=False)
    if path.exists() and path.read_text(encoding='utf-8-sig') == text:
        return '그대로'
    path.write_text(text, encoding='utf-8-sig')
    return '씀'

## 2. 입력

In [2]:
# ── 입력: 규칙 사전, LLM 사전, 제품군, 언급의 업종 ──
PM = pd.read_csv(GRAPH / 'product_mentions.csv', dtype=str).fillna('')
RULE = pd.read_csv(GRAPH / 'products.csv', dtype=str).fillna('').set_index('id')
MAP = pd.read_csv(LLM / 'llm_mention_map.csv', dtype=str).fillna('')
LP = pd.read_csv(LLM / 'llm_products.csv', dtype=str).fillna('')
LC = pd.read_csv(LLM / 'llm_candidates.csv', dtype=str).fillna('')
FAM = pd.read_csv(LLM / 'families_final.csv', dtype=str).fillna('').set_index('key')
co_sector = pd.read_csv(ROOT / 'source' / 'dart' / 'parsed' / 'companies.csv', dtype=str).fillna('').set_index('stock_code')['wi26_sector'].to_dict()
news = json.loads((ROOT / 'source' / 'news' / 'final' / 'news_final.json').read_text(encoding='utf-8'))['articles']
art_sectors = {aid: a['sectors'] for aid, a in news.items()}
assert not MAP.duplicated(MKEY).any(), 'LLM 표의 언급 열쇠가 겹친다'
print(f"규칙 언급 {len(PM):,}개(제품 {(PM['book'] == '제품').sum():,}, 브랜드 {(PM['book'] == '브랜드').sum():,}) / 규칙 제품 노드 {len(RULE):,}개")
print(f'LLM: 언급 {len(MAP):,}개, 제품 {len(LP):,}개, 후보 {len(LC):,}개, 제품군 {len(FAM)}개')

규칙 언급 29,831개(제품 23,127, 브랜드 6,704) / 규칙 제품 노드 13,767개
LLM: 언급 20,929개, 제품 11,693개, 후보 2,659개, 제품군 273개


## 3. 언급 → 제품

1. LLM 표에 있는 언급은 그 제품으로 잇는다(`how = LLM`).
2. 없는 언급은 규칙 노드로 대신 잇는다. 그 규칙 노드의 LLM 언급이 모두 같은 제품(들)으로 갔으면 그 제품(`규칙 노드로 대신`), 아니면 규칙 노드를 임시 제품 `PR:`로 둔다(`임시`).
3. 여러 제품으로 간 언급은 제품마다 한 줄이 된다. 브랜드 언급은 규칙 사전의 브랜드 노드 그대로다.

In [3]:
# ── 언급 → 제품: LLM 결과, 규칙 노드로 대신, 임시 제품 ──
pm = PM[PM['book'] == '제품'].merge(MAP, on=MKEY, how='left')
pm['how'] = pm['product_ids'].notna().map({True: 'LLM', False: ''})
llm_sets = pm[pm['how'] == 'LLM'].groupby('node_id')['product_ids'].agg(lambda s: set(s))
same_product = {n: next(iter(v)) for n, v in llm_sets.items() if len(v) == 1}     # 규칙 노드 → 그 노드의 LLM 언급이 모두 간 제품
miss = pm['how'] == ''
alt = miss & pm['node_id'].isin(same_product)
pm.loc[alt, 'product_ids'] = pm.loc[alt, 'node_id'].map(same_product)
pm.loc[alt, 'how'] = '규칙 노드로 대신'
temp = miss & ~alt
pm.loc[temp, 'product_ids'] = 'PR:' + pm.loc[temp, 'node_id'].str[2:]
pm.loc[temp, 'how'] = '임시'
pm['product_id'] = pm['product_ids'].map(split)
pm['multi'] = pm['product_id'].str.len() > 1                    # 여러 품목 언급(검색어에 원래 표기를 쓰지 않는다)
pm = pm.explode('product_id').drop(columns='product_ids')
br = PM[PM['book'] == '브랜드'].assign(product_id=lambda d: d['node_id'], how='브랜드(규칙)', multi=False)
PMF = pd.concat([pm, br], ignore_index=True)
print(f"제품 언급 {(PM['book'] == '제품').sum():,}개를 이은 방법", pm[~pm.index.duplicated()]['how'].value_counts().to_dict())
print(f"언급 → 제품 줄 {len(pm):,}개(여러 품목 언급 때문에 언급보다 {len(pm) - (PM['book'] == '제품').sum():,}줄 많다), 브랜드 {len(br):,}줄")
print('임시 제품이 된 언급의 출처', pm[pm['how'] == '임시']['source'].value_counts().to_dict(), f"/ 임시 제품 {pm.loc[pm['how'] == '임시', 'product_id'].nunique():,}개")

제품 언급 23,127개를 이은 방법 {'LLM': 21156, '임시': 1387, '규칙 노드로 대신': 584}
언급 → 제품 줄 24,105개(여러 품목 언급 때문에 언급보다 978줄 많다), 브랜드 6,704줄
임시 제품이 된 언급의 출처 {'news': 1387} / 임시 제품 1,144개


## 4. 제품 노드

| 단계 | ID | 무엇 |
|---|---|---|
| 제품군 | `PF:key` | 확정 제품군 중 제품이 하나라도 있는 것 |
| 제품 | `P:이름` | LLM 사전의 제품(중간 제품 포함) |
| 후보 | `PC:이름` | 후보 하위 제품. 언급이 없고 검색어로만 쓴다 |
| 임시 | `PR:규칙 키` | LLM 결과가 없는 언급의 규칙 노드 |

- 수(만드는 국내 상장사, DART 원재료 언급, 뉴스 기사, 언급)와 관련 업종(많이 나온 3개)은 그 제품에 직접 이어진 언급으로 센다.
- **검색어**: 영어 이름 → 언급의 영어 표기(많이 나온 순서, 여러 품목 언급은 빼고) → 규격 → 규칙 노드의 검색어(그 규칙 노드의 언급이 모두 이 제품으로 간 때만). 한글이 든 것은 뺀다. 후보는 LLM이 준 검색어. 검색어 노트북이 다시 고른다.
- **일반적인 제품**: 국내 상장사 10곳 이상이 만들거나, 통째로 들어온 규칙 노드가 일반적인 제품이었던 것.

In [4]:
# ── 제품 노드: 제품군·제품·후보·임시. 수·업종·검색어·일반적인지 ──
pm_p = PMF[PMF['book'] == '제품']
sec_of = lambda r: [co_sector.get(r.doc, '')] if r.source == 'dart' else art_sectors.get(r.doc, [])
pm_p = pm_p.assign(sectors=[sec_of(r) for r in pm_p.itertuples()])
stats = pd.DataFrame({
    'n_kr_makers': pm_p[(pm_p['source'] == 'dart') & (pm_p['field'] == '제품')].groupby('product_id')['doc'].nunique(),
    'n_dart_material': pm_p[pm_p['field'] == '원재료'].groupby('product_id').size(),
    'n_news': pm_p[pm_p['source'] == 'news'].groupby('product_id')['doc'].nunique(),
    'n_mentions': pm_p.groupby('product_id').size(),
    'sectors': pm_p.groupby('product_id')['sectors'].agg(lambda s: ' | '.join(x for x, _ in Counter(y for ys in s for y in ys if y).most_common(3)))})

# 규칙 노드가 통째로 한 제품으로 갔는지: 규칙 노드 → 그 제품
node_sets = pm.groupby('node_id')['product_id'].agg(lambda s: set(s))
whole = {n: next(iter(v)) for n, v in node_sets.items() if len(v) == 1}
whole_nodes = {}
for n, p in whole.items():
    whole_nodes.setdefault(p, []).append(n)
mention_terms = (pm_p[~pm_p['multi'].astype(bool)].assign(t=pm_p['name_en'].where(pm_p['name_en'] != '', pm_p['name']).str.strip())
                 .groupby(['product_id', 't']).size().rename('k').reset_index().sort_values(['product_id', 'k'], ascending=[True, False])
                 .groupby('product_id')['t'].agg(list).to_dict())


def terms_of(pid, name_en, specs):
    """검색어: 영어 이름, 언급 표기, 규격, 통째로 들어온 규칙 노드의 검색어. 한글 든 것은 빼고 대소문자 무시로 한 번씩."""
    out, seen = [], set()
    for t in [name_en] + mention_terms.get(pid, []) + split(specs) + [x for n in whole_nodes.get(pid, []) if n in RULE.index for x in split(RULE.at[n, 'search_terms'])]:
        t = re.sub(r'\s+', ' ', t).strip()
        if t and not HANGUL.search(t) and low(t) not in seen:
            seen.add(low(t))
            out.append(t)
    return ' | '.join(out[:MAX_TERMS])


fam_used = sorted(set(LP['family']))
rows = [dict(id='PF:' + k, level='제품군', name_en=FAM.at[k, 'name_en'], name_ko=FAM.at[k, 'name_ko'], family=k, kind=FAM.at[k, 'kind'],
             category=FAM.at[k, 'name_ko'], search_terms=FAM.at[k, 'name_en'], catch_all=FAM.at[k, 'catch_all'] == 'True') for k in fam_used]
for r in LP.itertuples():
    rows.append(dict(id=r.id, level='제품', name_en=r.name_en, name_ko=r.name_ko, family=r.family, kind=r.kind, category=FAM.at[r.family, 'name_ko'],
                     specs=r.specs, depth=int(r.depth), intermediate=r.intermediate == 'True', search_terms=terms_of(r.id, r.name_en, r.specs),
                     rule_nodes=r.rule_nodes))
for r in LC.itertuples():
    rows.append(dict(id=r.id, level='후보', name_en=r.name_en, name_ko=r.name_ko, family=r.family, category=FAM.at[r.family, 'name_ko'],
                     search_terms=' | '.join(dict.fromkeys([r.name_en] + split(r.search_terms))), candidate=True, reason=r.reason))
for pid in sorted(set(pm.loc[pm['how'] == '임시', 'product_id'])):
    n = 'P:' + pid[3:]
    x = RULE.loc[n]
    rows.append(dict(id=pid, level='임시', name_en=x['name_en'], name_ko=x['name_ko'], category=x['category'], search_terms=x['search_terms'],
                     pending_llm=True, rule_nodes=n))
NODES = pd.DataFrame(rows).join(stats, on='id')
for c in ['n_kr_makers', 'n_dart_material', 'n_news', 'n_mentions', 'depth']:
    NODES[c] = NODES[c].fillna(0).astype(int)
for c in ['intermediate', 'candidate', 'pending_llm', 'catch_all']:
    NODES[c] = NODES[c].eq(True)
NODES = NODES.fillna('')
rule_generic = {n for n in RULE.index if RULE.at[n, 'generic'] == 'True'}
NODES['generic'] = (NODES['n_kr_makers'] >= GENERIC_MAKERS) | NODES['id'].map(lambda p: any(n in rule_generic for n in whole_nodes.get(p, [])))
NODES.loc[NODES['level'] == '임시', 'generic'] = NODES.loc[NODES['level'] == '임시', 'rule_nodes'].isin(rule_generic)
assert NODES['id'].is_unique, '제품 노드 ID가 겹친다'
assert set(pm['product_id']) <= set(NODES['id']), '언급이 가리키는 제품이 노드 표에 없다'
print(NODES['level'].value_counts().to_dict(), f"/ 일반적인 제품 {NODES['generic'].sum()}개")

{'제품': 11693, '후보': 2659, '임시': 1144, '제품군': 273} / 일반적인 제품 101개


## 5. 계층

하위 → 상위 한 줄이 그래프의 `SUBTYPE_OF` 관계 하나다. 제품은 상위 제품, 맨 위 제품은 제품군, 후보는 상위 제품으로 잇는다. 임시 제품은 계층이 없다. 계층에는 근거 문장이 없으므로 사용자에게 보이는 경로에서는 "분류"로 표시한다.

In [5]:
# ── 계층: 제품 → 상위 제품(없으면 제품군), 후보 → 상위 제품 ──
H = pd.concat([LP.assign(child=LP['id'], parent=[p or 'PF:' + f for p, f in zip(LP['parent'], LP['family'])], child_level='제품'),
               LC.assign(child=LC['id'], parent=LC['parent'], child_level='후보')])[['child', 'parent', 'child_level']]
assert set(H['parent']) <= set(NODES['id']), '상위가 노드 표에 없다'
assert not H.duplicated('child').any()
print(f"계층 {len(H):,}줄: 제품 → 상위 제품 {(LP['parent'] != '').sum():,}, 제품 → 제품군 {(LP['parent'] == '').sum():,}, 후보 → 제품 {len(LC):,}")

계층 14,352줄: 제품 → 상위 제품 6,037, 제품 → 제품군 5,656, 후보 → 제품 2,659


## 6. 브랜드 → 제품

- 브랜드 언급(출처, 문서, 기사 개체 번호, 이름)에 LLM 3단계 결과를 붙인다. 결과가 없는 언급(관계에 쓰이지 않은 뉴스 브랜드, 새로 들어온 브랜드)은 잇지 않는다.
- 언급마다 이을 곳: 코드가 정한 제품, 없으면 제품군(`PF:`), 제품군도 없으면(여러 제품에 쓰인 회사 이름 등) 잇지 않는다.
- 브랜드 노드마다 이을 곳을 모은다. 같은 브랜드가 회사마다 다른 제품이면(HYUNDAI: 굴착기, 지게차) 여러 개다. 같은 브랜드가 어떤 제품에 이어졌으면 그 제품의 제품군으로만 이은 것은 뺀다.
- 브랜드 종류(`brand_kind`)는 그 브랜드 언급에서 가장 많이 나온 종류다.

In [6]:
# ── 브랜드 → 제품: 3단계 결과를 브랜드 언급에 붙이고, 브랜드 노드마다 이을 제품(또는 제품군)을 모은다 ──
BKEY = ['source', 'doc', 'ref', 'name']
S3 = pd.read_csv(BRAND_FILE, dtype=str).fillna('')
assert not S3.duplicated(BKEY).any(), '3단계 결과의 언급 열쇠가 겹친다'
pb = PM[PM['book'] == '브랜드'].merge(S3[BKEY + ['kind', 'family', 'product', 'how']], on=BKEY, how='left').fillna('')
pb['target'] = [p or ('PF:' + f if f else '') for p, f in zip(pb['product'], pb['family'])]
print(f"브랜드 언급 {len(pb):,}개 중 3단계 결과 있음 {(pb['kind'] != '').sum():,}개(없는 것: {pb.loc[pb['kind'] == '', 'source'].value_counts().to_dict()})")
print('이을 곳', {'제품': int((pb['product'] != '').sum()), '제품군만': int(((pb['product'] == '') & (pb['family'] != '')).sum()), '없음': int(((pb['kind'] != '') & (pb['target'] == '')).sum())})
assert set(pb.loc[pb['target'] != '', 'target']) <= set(NODES['id']), '브랜드가 가리키는 제품이 노드 표에 없다'
BP = (pb[pb['target'] != ''].groupby(['node_id', 'target'])
      .agg(n_mentions=('doc', 'size'), how=('how', lambda s: s.value_counts().index[0])).reset_index().rename(columns={'node_id': 'brand'}))
fam_of = NODES.set_index('id')['family']
covered = {(b, fam_of[t]) for b, t in zip(BP['brand'], BP['target']) if not t.startswith('PF:')}
drop = [t.startswith('PF:') and (b, t[3:]) in covered for b, t in zip(BP['brand'], BP['target'])]
BP = BP[[not d for d in drop]].copy()
BP['target_level'] = BP['target'].str.startswith('PF:').map({True: '제품군', False: '제품'})
BRANDS = pd.read_csv(GRAPH / 'brands.csv', dtype=str).fillna('')
kind_of = pb[pb['kind'] != ''].groupby('node_id')['kind'].agg(lambda s: s.value_counts().index[0])
BRANDS['brand_kind'] = BRANDS['id'].map(kind_of).fillna('')
per = BP.groupby('brand').size()
print(f"브랜드 노드 {len(BRANDS):,}개 중 이은 것 {BP['brand'].nunique():,}개(제품 {BP.loc[BP['target_level'] == '제품', 'brand'].nunique():,}, 제품군만 "
      f"{BP['brand'].nunique() - BP.loc[BP['target_level'] == '제품', 'brand'].nunique():,}) / 관계 {len(BP):,}줄, 이을 곳이 둘 이상인 브랜드 {(per > 1).sum()}개 "
      f"(같은 제품군으로만 이은 것 {sum(drop)}줄 뺌)")
print('브랜드 종류', BRANDS['brand_kind'].value_counts().to_dict())

브랜드 언급 6,704개 중 3단계 결과 있음 5,838개(없는 것: {'news': 866})
이을 곳 {'제품': 4880, '제품군만': 738, '없음': 220}


브랜드 노드 5,596개 중 이은 것 4,818개(제품 4,203, 제품군만 615) / 관계 4,975줄, 이을 곳이 둘 이상인 브랜드 153개 (같은 제품군으로만 이은 것 51줄 뺌)
브랜드 종류 {'제품 브랜드': 4290, '': 594, '서비스 브랜드': 517, '회사·그룹 이름': 108, '브랜드 아님': 87}


## 7. 저장

In [7]:
# ── 저장: 내용이 바뀔 때만 쓴다 ──
NODE_COLS = ['id', 'level', 'name_en', 'name_ko', 'family', 'kind', 'category', 'depth', 'specs', 'intermediate', 'candidate', 'pending_llm', 'catch_all',
             'search_terms', 'n_kr_makers', 'n_dart_material', 'n_news', 'n_mentions', 'sectors', 'generic', 'rule_nodes', 'reason']
print('products_final.csv', save_csv(NODES[NODE_COLS].sort_values('id'), GRAPH / 'products_final.csv'),
      '/ product_mentions_final.csv', save_csv(PMF[list(PM.columns) + ['product_id', 'how']], GRAPH / 'product_mentions_final.csv'),
      '/ product_hierarchy.csv', save_csv(H.sort_values('child'), GRAPH / 'product_hierarchy.csv'),
      '/ brands_final.csv', save_csv(BRANDS, GRAPH / 'brands_final.csv'),
      '/ brand_products.csv', save_csv(BP[['brand', 'target', 'target_level', 'n_mentions', 'how']].sort_values(['brand', 'target']), GRAPH / 'brand_products.csv'))

products_final.csv 그대로 / product_mentions_final.csv 그대로 / product_hierarchy.csv 그대로 / brands_final.csv 그대로 / brand_products.csv 그대로


## 8. 확인

- 웨이퍼 언급이 어디로 갔는지(섞인 노드 나누기), 메모리 계층과 검색어.
- 브랜드 → 제품 예: 뉴스의 GLP-1 브랜드가 이어진 제품과, 그 제품을 DART에 적은 국내 회사 수.
- **뉴스와 DART를 잇는 제품 수**(성공 기준). 규칙 사전은 277개였다. 직접(같은 제품에 둘 다 언급)과, 계층으로(뉴스 언급 제품과 DART 언급 제품이 같은 조상·자손 줄에 있음)를 센다. 제품군 노드는 넓어서 빼고 센다.

In [8]:
# ── 확인: 섞인 노드, 계층·검색어 예, 뉴스와 DART를 잇는 제품 수 ──
print('■ 규칙 노드 P:wafer의 언급')
display(pm[pm['node_id'] == 'P:wafer'].groupby(['product_id', 'source']).size().rename('언급').to_frame())
N = NODES.set_index('id')
print('■ DRAM 아래 제품과 후보, 검색어')
kids = H[H['parent'] == 'P:dram']['child']
display(N.loc[['P:dram'] + list(kids), ['level', 'name_ko', 'specs', 'n_kr_makers', 'n_news', 'search_terms']])

up = dict(zip(H['child'], H['parent']))
def chain(p):
    out = []
    while p in up and not up[p].startswith('PF:'):
        p = up[p]
        out.append(p)
    return out
prod = NODES[NODES['level'] == '제품']
has_news = set(prod.loc[prod['n_news'] > 0, 'id'])
has_dart = set(prod.loc[(prod['n_kr_makers'] + prod['n_dart_material']) > 0, 'id'])
direct = has_news & has_dart
via = {p for p in has_news if any(a in has_dart for a in chain(p))} | {a for p in has_news for a in chain(p) if a in has_dart}
via |= {p for p in has_dart if any(a in has_news for a in chain(p))}
print(f'뉴스와 DART를 잇는 제품: 직접 {len(direct):,}개, 계층을 거쳐서만 {len(via - direct):,}개 (규칙 사전 277개)')
display(prod[prod['id'].isin(direct)].sort_values('n_news', ascending=False).head(15)[['id', 'name_ko', 'family', 'n_kr_makers', 'n_news']])
print('■ 브랜드 → 제품 예')
ex = BP[BP['brand'].isin(['B:zepbound', 'B:mounjaro', 'B:ozempic', 'B:wegovy', 'B:shin ramyun', 'B:cybertruck', 'B:hyundai', 'B:creora'])]
display(ex.join(N[['name_ko', 'n_kr_makers', 'n_news']], on='target'))
print('■ 브랜드가 많이 이어진 제품')
display(BP.groupby('target').size().sort_values(ascending=False).head(12).rename('브랜드 수').to_frame().join(N[['name_ko', 'family']]))

■ 규칙 노드 P:wafer의 언급


,,언급
product_id,source,
P:semiconductor wafers,dart,12
P:solar wafers,dart,1
P:wafer biscuits,dart,2


■ DRAM 아래 제품과 후보, 검색어


,level,name_ko,specs,n_kr_makers,n_news,search_terms
id,,,,,,
P:dram,제품,디램,,4,3,DRAM | Dynamic Random-Access Memory | dynamic random access memory
P:high-bandwidth memory,제품,고대역폭 메모리,HBM4,2,9,high-bandwidth memory | High Bandwidth Memory | HBM4 | high bandwidth memory...
P:low-power dram,제품,저전력 D램,LPDDR4 | LPDDR4X | LPDDR5X,2,1,low-power DRAM | LPDDR5X | LPDDR4 | LPDDR4X | low-power DDR4 | LPDRAM | low ...
P:ddr sdram,제품,DDR 동기식 D램,DDR4 | DDR5,3,1,DDR SDRAM | DDR5 | DDR5 memory module | consumer DDR4 | consumer DDR5 | DDR4...
P:cellular ram,제품,셀룰러 RAM,,1,0,cellular RAM | cellularRAM | CRAM
P:consumer dram,제품,소비자용 DRAM,,1,0,consumer DRAM | consumer memory DRAM
P:graphics dram,제품,그래픽 DRAM,GDDR7,1,0,graphics DRAM | GDDR7 | GDDR7 memory
PC:embedded dram,후보,임베디드 DRAM,,0,0,embedded DRAM | eDRAM
PC:reduced-latency dram,후보,저지연 DRAM,,0,0,reduced-latency DRAM | reduced latency DRAM | RLDRAM


뉴스와 DART를 잇는 제품: 직접 295개, 계층을 거쳐서만 1,399개 (규칙 사전 277개)


,id,name_ko,family,n_kr_makers,n_news
6888,P:crude oil,원유,crude_oil,6,48
2705,P:natural gas,천연가스,natural_gas_and_lpg,13,39
2706,P:liquefied natural gas,액화천연가스,natural_gas_and_lpg,3,33
7341,P:ai chips,AI 반도체,ai_accelerator_chips,2,29
5013,P:diesel fuel,경유,refined_petroleum_fuels,9,27
3587,P:steel products,철강 제품,steel_products,15,23
5015,P:gasoline,휘발유,refined_petroleum_fuels,9,20
11026,P:electricity,전력,electricity_and_heat_supply,28,16
2609,P:memory chips,메모리 반도체,memory_chips,3,15
3589,P:hot-rolled steel sheet and coil,열연강판 및 코일,steel_products,5,12


■ 브랜드 → 제품 예


,brand,target,n_mentions,how,target_level,name_ko,n_kr_makers,n_news
875,B:creora,P:spandex fiber,2,블록 제품 번호,제품,스판덱스 섬유,3,0
876,B:creora,P:spandex yarn,2,블록 제품 번호,제품,스판덱스 원사,2,0
1859,B:hyundai,P:automotive motors,1,블록 제품 번호,제품,자동차용 모터,2,0
1860,B:hyundai,P:automotive parts,1,이름으로 찾음,제품,자동차 부품,20,0
1861,B:hyundai,P:forklifts,1,블록 제품 번호,제품,지게차,6,0
2681,B:mounjaro,P:glp-1 receptor agonists,5,넓은 이름으로 찾음,제품,GLP-1 수용체 작용제,4,11
3042,B:ozempic,P:glp-1 receptor agonists,3,블록 제품 번호,제품,GLP-1 수용체 작용제,4,11
3655,B:shin ramyun,P:instant noodles,3,블록 제품 번호,제품,라면,9,0
4436,B:wegovy,P:glp-1 receptor agonists,5,이름으로 찾음,제품,GLP-1 수용체 작용제,4,11
4581,B:zepbound,P:glp-1 receptor agonists,6,넓은 이름으로 찾음,제품,GLP-1 수용체 작용제,4,11


■ 브랜드가 많이 이어진 제품


,브랜드 수,name_ko,family
target,,,
P:video games,107,비디오 게임,video_games
P:skin care products,93,기초화장품,cosmetics_and_personal_care
P:diabetes drugs,85,당뇨병 치료제,diabetes_and_obesity_drugs
P:apparel,82,의류,apparel_and_accessories
P:mobile games,62,모바일 게임,video_games
P:cosmetics,47,화장품,cosmetics_and_personal_care
P:antihypertensive drugs,47,고혈압 치료제,cardiovascular_and_blood_disorder_drugs
P:cardiovascular drugs,44,심혈관 치료제,cardiovascular_and_blood_disorder_drugs
P:snack foods,42,스낵류,bakery_and_confectionery
